In [57]:
######################
# 0. 코랩 환경 준비
from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics opencv-python

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [58]:
######################
# 1. YOLOv8 포츠 추정 모델 불러오기
from ultralytics import YOLO

model = YOLO("/content/drive/MyDrive/models/yolov8m-pose.pt")

In [59]:
######################
# 2. 모델 추론
import torch

def predict(frame, iou=0.7, conf=0.25):
    results = model(
        source=frame,
        device="0" if torch.cuda.is_available() else "cpu",
        iou=0.7,
        conf=0.25,
        verbose=False,
    )
    result = results[0]
    return result

In [60]:
######################
# 3. 경계 상자 시각화
def draw_boxes(result, frame):
    for boxes in result.boxes:
        x1, y1, x2, y2, score, classes = boxes.data.squeeze().cpu().numpy()
        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 0, 255), 1)
    return frame

In [61]:
######################
# 4. 키 포인트 시각화 (Colab + 최신 Ultralytics)
from ultralytics.utils.plotting import Annotator
from google.colab.patches import cv2_imshow
import cv2

def draw_keypoints(result, frame, show=False):
    annotator = Annotator(frame, line_width=2)

    # keypoints 없을 때 대비
    if result.keypoints is None or len(result.keypoints) == 0:
        if show:
            cv2_imshow(frame)
        return frame

    # 전체 keypoints tensor (N, K, 3)
    kps_all = result.keypoints.data

    for person_kps in kps_all:
        # skeleton 포함 기본 그리기
        annotator.kpts(person_kps)

        nkps = person_kps.cpu().numpy()

        for idx, (x, y, score) in enumerate(nkps):
            if score > 0.3:   # threshold 낮추는 게 중요
                cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1)
                cv2.putText(
                    frame,
                    str(idx),
                    (int(x), int(y)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 0),
                    1
                )

    # Colab 출력 옵션
    if show:
        cv2_imshow(frame)

    return frame

In [69]:
######################
# 5. 비디오 파일 저장하기
import cv2

fourcc = cv2.VideoWriter_fourcc(*'mp4v')
out = cv2.VideoWriter('/content/drive/MyDrive/output/output.mp4', fourcc, 30,
                      (int(capture.get(3)), int(capture.get(4))))

capture = cv2.VideoCapture("/content/drive/MyDrive/datasets/airport.mp4")

while True:
    ret, frame = capture.read()
    if not ret:
        break

    results = predict(frame)

    for result in results:
        frame = draw_boxes(result, frame)
        frame = draw_keypoints(result, frame)

    out.write(frame)

capture.release()
out.release()